[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day13_live.ipynb)

# Day 13 · 강의 — Docker 다루기

Codex 에게 시킬 것을 정하고, 받은 파일을 읽고 고친다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

모든 셀에 **코드가 채워져 있다.** 위에서부터 실행해 결과를 눈으로 확인한다.
강사가 설명하는 동안 값을 바꿔 가며 돌려 본다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 이 노트북이 하는 일

Docker 는 **Colab 에서 안 돌아간다.** 실제 명령은 **터미널에서** 친다.

이 노트북은 그 앞뒤를 맡는다 &mdash;
**무엇을 시킬지 정하고**, Codex 가 내놓은 파일을 **읽고 고치는** 연습이다.

> 코드를 쓰는 문제는 없다. 고를 것을 고르고 틀린 데를 짚는 문제다.

In [ ]:
# 검사에 쓸 도구만 준비한다. 설치할 것이 없다.
import re, textwrap

def 줄번호(글, 찾을것):
    for i, line in enumerate(글.strip().split('\n'), 1):
        if 찾을것 in line:
            return i
    return None

print('준비됐다')

## 2. 여섯 칸 적기

Codex 에게 시키기 전에 **여섯 가지**를 정해야 한다. 이걸 안 주면 지어낸다.

## 3. 프롬프트로 옮기기

여섯 칸을 **문장으로** 바꾼다. 아래 뼈대에 값만 끼워 넣으면 된다.

**왜 이 두 줄인가.**

`0.0.0.0` &mdash; 안 적으면 상자는 뜨는데 밖에서 못 붙는다. 제일 잦은 사고다.

`값은 적지 마라` &mdash; 안 적으면 키를 `ENV` 로 이미지에 박아 넣는다. 지워도 남는다.

> **실습문제 1.** 여섯 칸을 프롬프트로 만든다. **빠뜨리면 안 되는 두 줄**이 뒤에 있다.
> 그 두 줄이 없으면 Codex 가 늘 같은 실수를 한다.

In [ ]:
# 여섯칸 을 그대로 쓴다
프롬프트 = f'''Dockerfile 을 만들어 줘.

{여섯칸['언어와 버전']} 을 쓴다. 바탕 이미지는 slim 판으로.
꾸러미는 {여섯칸['라이브러리']}
시작 명령은 {여섯칸['시작 명령']}
{여섯칸['여는 포트']} 포트를 연다

앱은 127.0.0.1 이 아니라 0.0.0.0 으로 열게 한다
키는 환경변수로 받기만 한다. 값은 적지 마라
'''
print(프롬프트)

assert '0.0.0.0' in 프롬프트, '여는 주소를 못 박아야 한다'
assert '값은 적지' in 프롬프트 or '값을 적지' in 프롬프트, '키 값을 적지 말라고 해야 한다'
print()
print('두 줄이 들어갔다. 이 프롬프트를 그대로 Codex 에 붙여 넣는다.')

## 4. 틀린 Dockerfile 찾기

Codex 가 내놓은 것을 **그냥 쓰지 않는다.** 아래 셋을 차례로 본다.

In [ ]:
# 검토할 Dockerfile 셋. 일부러 틀린 곳을 넣어 두었다.
후보 = [
    '바탕 이미지에 버전을 안 적었다',
    '키 값을 이미지에 박아 넣었다',
    '앱이 127.0.0.1 로 열려 밖에서 못 붙는다',
    '폴더를 통째로 복사해 잡동사니가 들어간다',
    '꾸러미를 코드보다 나중에 깔아 빌드가 느리다',
    '데이터를 이미지에 넣어 상자를 지우면 사라진다',
    '틀린 곳이 없다',
]
for i, x in enumerate(후보, 1):
    print('%d. %s' % (i, x))

In [ ]:
D1 = '''
FROM python:latest
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
ENV NVIDIA_API_KEY=nvapi-abc123
EXPOSE 8000
CMD ["python", "app.py"]
'''
print(D1)

In [ ]:
D2 = '''
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY cell_process.csv .
EXPOSE 8000
CMD ["python", "app.py"]
'''
print(D2)

**여기서 한 가지 더.** `D2` 는 앱이 `0.0.0.0` 으로 여는지 **Dockerfile 만 봐서는 모른다.**
그건 `app.py` 안에 있다. 그래서 프롬프트에 미리 못 박아 두는 것이다.

> **실습문제 2.** 위 `D1` 에서 틀린 곳을 **넷** 고른다. 번호로 적는다.
> 후보 목록에서 고른다. 순서는 상관없다.

In [ ]:
# 후보 번호를 넣는다
답1 = [1, 2, 4, 5]

정답 = {1, 2, 4, 5}
assert set(답1) == 정답, '다시 본다. 힌트 — latest · ENV · COPY . . · pip 순서'
for n in sorted(답1):
    print('%d. %s' % (n, 후보[n-1]))
print()
print('넷 다 찾았다. 이 넷이 실제로 제일 자주 나온다.')

## 5. compose.yml 읽기

서비스가 여럿이면 **무엇이 밖으로 열리는지**가 제일 중요하다.

In [ ]:
C1 = '''
services:
  web:
    build: ./web
    ports: ["3000:3000"]
    environment:
      API_URL: http://api:8000

  api:
    build: ./api
    ports: ["8000:8000"]

  db:
    image: postgres:16
    ports: ["5432:5432"]
    environment:
      POSTGRES_PASSWORD: mypassword
'''
print(C1)

## 6. 폐쇄망용으로 고치기

바깥이 막힌 곳으로 넘길 때는 **`compose.yml` 을 따로 만든다.**
`build:` 를 그대로 두면 받는 쪽에서 다시 만들려고 인터넷을 찾는다.

## 7. 터미널에서 할 것

여기까지가 노트북에서 하는 일이다. **아래는 터미널에서 친다.**
노트북에서는 안 돌아가니 눈으로 읽고 순서만 익혀 둔다.

**만드는 쪽**

```bash
# 4단계 · 이미지를 만든다
docker compose build
docker images                        # posco-web · posco-api 가 보이나

# 5단계 · 띄워서 확인한다
docker compose up -d
docker compose ps                    # 둘 다 Up 인가
curl http://127.0.0.1:3000/          # 화면이 나오나
```

**옮기는 쪽**

```bash
# 6단계 · 파일 하나로 뽑는다
docker save posco-web:1.0 posco-api:1.0 -o images.tar
ls -lh images.tar                    # 100MB 안팎이면 정상

# 7단계 · 옮긴 곳에서 띄운다 (인터넷 안 씀)
docker load -i images.tar
docker compose up -d
```

**막히면 보는 순서**

```bash
docker compose ps                    # 떴나
docker compose logs web --tail 20    # 왜 안 되나
lsof -nP -iTCP:3000 -sTCP:LISTEN     # 포트를 누가 쥐고 있나
```

### 오늘 손에 남는 것

**하나** &mdash; Codex 에게 시키기 전에 **여섯 칸**을 정한다. 안 주면 지어낸다.

**둘** &mdash; 프롬프트에 **`0.0.0.0`** 과 **「값은 적지 마라」** 두 줄을 반드시 넣는다.

**셋** &mdash; 받은 Dockerfile 은 **버전 · 키 · COPY 범위 · 순서** 넷을 본다.

**넷** &mdash; `compose.yml` 에서 <b>ports 를 적은 것만</b> 밖으로 열린다. DB 에는 안 적는다.

**다섯** &mdash; 폐쇄망용은 **`build:` 를 `image:` 로** 바꾼 파일을 따로 둔다.

**여섯** &mdash; 넘길 것은 **`images.tar` · `compose.yml` · `env.example`** 셋이다.